In [3]:
import pandas as pd
import numpy as np

In [4]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print(PROJECT_ROOT)

d:\AI Projects\Credit-Risk-Knowledge-Assistant


In [5]:
from app.services.document_loader import (
    DOCUMENT_DIR,
    load_all_pdfs
)

from app.services.text_splitter import (
    split_documents
)

from app.services.embedding_service import (
    get_embedding_model
)



d:\AI Projects\Credit-Risk-Knowledge-Assistant\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
from langchain_chroma import Chroma

In [ ]:
documents = load_all_pdfs(DOCUMENT_DIR)

chunks = split_documents(
    documents,
    chunk_size=1000,
    chunk_overlap=200)

print("Page documents:", len(documents))
print("Chunks:", len(chunks))

In [7]:
embedding_model = get_embedding_model()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11625.14it/s]


In [8]:
CHROMA_DIR = PROJECT_ROOT / "chroma_db"

COLLECTION_NAME = "credit_risk_knowledge"

print(CHROMA_DIR)

d:\AI Projects\Credit-Risk-Knowledge-Assistant\chroma_db


In [ ]:
import hashlib


def create_chunk_id(chunk):

    identity = (
        f"{chunk.metadata.get('source_file')}|"
        f"{chunk.metadata.get('page_number')}|"
        f"{chunk.metadata.get('chunk_id')}|"
        f"{chunk.page_content}"
    )

    return hashlib.sha256(
        identity.encode("utf-8")
    ).hexdigest()

In [ ]:
chunk_ids = [
    create_chunk_id(chunk)
    for chunk in chunks
]

In [ ]:
chunk_ids

### experiment with only 50 chunks

In [ ]:
sample_chunks = chunks[:50]

sample_ids = chunk_ids[:50]

In [ ]:
vector_store = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embedding_model,
    persist_directory=str(CHROMA_DIR),

    collection_configuration={
        'hnsw':{
        'space':'cosine'
    }
    }
)

In [ ]:
## Add the sample chunks

stored_ids = vector_store.add_documents(
    documents=sample_chunks,
    ids = sample_ids
)

print("Stored :", len(stored_ids))

In [ ]:
##Inspect what's stored

stored_data = vector_store.get(
    limit=3
)

stored_data

In [ ]:
stored_data.keys()

In [ ]:
stored_data["ids"]

In [ ]:
stored_data["documents"]

In [ ]:
question = "What is the objective of Ind AS 109?"

results = vector_store.similarity_search(
    query=question,
    k=5
)

In [ ]:
for rank, doc in enumerate(results, start=1):

    print("=" * 80)

    print("Rank:", rank)

    print(
        "Source:",
        doc.metadata.get("source_file")
    )

    print(
        "Page:",
        doc.metadata.get("page_number")
    )

    print()

    print(doc.page_content[:700])

In [ ]:
results_with_scores = (
    vector_store.similarity_search_with_score(
        query=question,
        k=5
    )
)

In [ ]:
type(results_with_scores)

In [ ]:
results_with_scores

In [ ]:
for rank, (doc, score) in enumerate(
    results_with_scores,
    start=1
):

    print("=" * 80)

    print("Rank:", rank)

    print("Distance:", round(score, 4))

    print(
        "Source:",
        doc.metadata.get("source_file")
    )

    print(
        "Page:",
        doc.metadata.get("page_number")
    )

    print()

    print(doc.page_content[:500])

In [ ]:
#Check metadata filtering

results = vector_store.similarity_search(
    query="expected credit loss",
    k=5,
    filter={
        "source_file": "INDAS109.pdf"
    }
)

In [ ]:
from pathlib import Path
import shutil

# Path to your Chroma database folder
CHROMA_DIR = Path(r"D:\AI Projects\Credit-Risk-Knowledge-Assistant\chroma_db")

# Remove old Chroma database if it exists
if CHROMA_DIR.exists():
    shutil.rmtree(CHROMA_DIR)
    print("Old Chroma database removed.")
else:
    print("Chroma database folder does not exist.")

In [ ]:
vector_store = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embedding_model,
    persist_directory=str(CHROMA_DIR),
    collection_configuration={
        "hnsw": {
            "space": "cosine"
        }
    }
)

In [ ]:
chunk_ids = [
    create_chunk_id(chunk)
    for chunk in chunks
]

In [ ]:
stored_ids = vector_store.add_documents(
    documents=chunks,
    ids=chunk_ids
)

print(
    "Total chunks stored:",
    len(stored_ids)
)

In [9]:
embedding_model = get_embedding_model()

vector_store = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embedding_model,
    persist_directory=str(CHROMA_DIR)
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8039.10it/s]


In [10]:
results = vector_store.similarity_search(
    "What is the objective of Ind AS 109?",
    k=5
)

In [11]:
results

[Document(id='44023ecf7a9d611cd8b763d1a45819847b3daf67460032d1cc63b0851ff1c696', metadata={'creator': 'Microsoft® Word 2013', 'page_number': 188, 'producer': 'Microsoft® Word 2013', 'author': 'nidhi1', 'moddate': '2015-02-20T10:47:32+05:30', 'total_pages': 189, 'page_label': '188', 'creationdate': '2015-02-20T10:47:32+05:30', 'chunk_length': 721, 'source_file': 'INDAS109.pdf', 'page': 187, 'chunk_id': 621, 'source': 'D:\\AI Projects\\Credit-Risk-Knowledge-Assistant\\documents\\INDAS109.pdf'}, page_content='433 \n \nAppendix E \nReferences to matters contained in other Indian \nAccounting Standards \nThis appendix is an integral part of the Ind AS. \n \nThis appendix lists the appendices which are part of other Indian Accounting \nStandards and make reference to Ind AS 109,Financial Instruments. \n1. Appendix A,Rights to Interests arising from Decommissioning,Restoration \nand Environmental Rehabilitation contained in Ind AS 37 , Provisions, \nContingent Liabilities and Contingent Asset